<a href="https://colab.research.google.com/github/look4pritam/LargeLanguageModels/blob/master/Notebooks/Inference/Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 46.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

model_id = "microsoft/Phi-3-mini-4k-instruct"

In [ ]:
config = AutoConfig.from_pretrained(model_id)
if hasattr(config, "rope_scaling") and config.rope_scaling is not None:
    if "type" not in config.rope_scaling and "rope_type" in config.rope_scaling:
        config.rope_scaling["type"] = config.rope_scaling["rope_type"]


In [ ]:
model1 = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=False
)

In [5]:
print(f"## Model Type: {type(model1).__name__}")
mem_bytes = model1.get_memory_footprint()
mem_gb = mem_bytes / (1024**3)

print(f"## Memory Footprint: {mem_gb:.2f} GB")

num_params = sum(p.numel() for p in model1.parameters())
print(f"## Parameter Count: {num_params / 1e9:.2f} Billion")


## Model Type: Phi3ForCausalLM
## Memory Footprint: 7.12 GB
## Parameter Count: 3.82 Billion


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [7]:
input_text = "Instruct: Write a short poem about a cat.\nOutput:"
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

outputs = model1.generate(**inputs, max_new_tokens=550)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Instruct: Write a short poem about a cat.
Output:

Whiskers twitching, eyes gleaming bright,
A feline grace in the soft moonlight.
Silent paws tread on the carpet's lace,
A gentle purr, a warm embrace.

Curled up in a sunbeam's glow,
A cat's contentment, a peaceful show.
With a stretch and yawn, the day begins,
A playful spirit, a life that spins.

A creature of mystery, a silent friend,
A cat's companionship, a bond that won't end.
In every meow, a story unfolds,
A life of adventure, a tale to be told.
